# Business value of TRM timing decisions

This report compares persistence with the ARIMA order selected on validation MAE. The USD amount is an explicit scenario input, not a project-wide assumption. The test partition is evaluated only after the order is frozen.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from trm_forecast.business.evaluation import evaluate_business_value
from trm_forecast.evaluation.arima import evaluate_arima
from trm_forecast.evaluation.persistence import evaluate_persistence
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data/interim/trm_normalized.csv').exists())
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
evaluation = config['evaluation']
data = pd.read_csv(ROOT / 'data/interim/trm_normalized.csv', parse_dates=['date'])
series = data.set_index('date')['trm'].sort_index()
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
protocol = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs)
result = protocol.split(series)
horizons = tuple(evaluation['horizons'])
usd_amount = 100_000  # scenario input; change for the operation being evaluated

In [ ]:
# Select the ARIMA order using validation only, then freeze it for test.
candidate_rows = []
for order in config['arima']['candidate_orders']:
    _, metrics = evaluate_arima(result, tuple(order), horizons, ('validation',), config['arima']['min_history'], config['arima']['maxiter'], config['arima']['refit_frequency'])
    candidate_rows.append({'order': tuple(order), 'validation_mae': metrics['mae'].mean()})
selection = pd.DataFrame(candidate_rows).sort_values(['validation_mae', 'order'])
selected_order = selection.iloc[0]['order']
selection

In [ ]:
persistence, _ = evaluate_persistence(result, horizons)
persistence = persistence.rename(columns={'prediction': 'prediction'})
arima, _ = evaluate_arima(result, tuple(selected_order), horizons, ('validation', 'test'), config['arima']['min_history'], config['arima']['maxiter'], config['arima']['refit_frequency'])
level_forecasts = pd.concat([persistence, arima], ignore_index=True)
level_forecasts = level_forecasts[level_forecasts['status'].isna() | level_forecasts['status'].isin(['converged', 'converged_with_warning'])]
decisions, business_summary = evaluate_business_value(level_forecasts, usd_amount=usd_amount, horizons=horizons, strict=False)
business_summary.sort_values(['partition', 'operation', 'model'])

Interpret positive gross COP benefit as improvement over executing at horizon 1. Treat validation results as the basis for any model or policy choice; use the test rows only as the final frozen evaluation. This gross comparison excludes transaction costs, spreads, execution constraints, and forecast availability risk.